In [1]:
# TASK 4 - OPTIMIZATION MODEL
# Intern: Vivek Dipak Kshirsagar | ID: CTIS6665
# Codtech IT Solutions | Data Science Internship

import sys
!{sys.executable} -m pip install pulp

print("✅ PuLP Installed!")

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.4 MB 593.4 kB/s eta 0:00:27
   - -------------------------------------- 0.8/16.4 MB 752.1 kB/s eta 0:00:21
   - -------------------------------------- 0.8/16.4 MB 752.1 kB/s eta 0:00:21
   - -------------------------------------- 0.8/16.4 MB 752.1 kB/s eta 0:00:21
   -- ------------------------------------- 1.0/16.4 MB 654.3 kB/s eta 0:00:24
   -- ------------------------------------- 1.0/16.4 MB 654.3 kB/s eta 0:00:24
   --- ------------------------------------ 1.3/16.4 MB 588.8 kB/s eta 0:00:26
   --- ------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
# STEP 2 - IMPORT LIBRARIES

import pulp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ All Libraries Imported!")
print(f"PuLP Version: {pulp.__version__}")

✅ All Libraries Imported!
PuLP Version: 3.3.0


In [3]:
# STEP 3 - BUSINESS PROBLEM SETUP
# Problem: A factory makes Product A and Product B
# Maximize Profit!

print("="*50)
print("BUSINESS PROBLEM: Factory Production Optimization")
print("="*50)
print("""
A factory produces two products: A and B

Product A:
- Profit per unit: Rs. 500
- Machine hours required: 2
- Labor hours required: 4

Product B:
- Profit per unit: Rs. 400
- Machine hours required: 5
- Labor hours required: 3

Constraints:
- Total Machine hours available: 40
- Total Labor hours available: 36
- Max Product A: 10 units
- Max Product B: 8 units

GOAL: Maximize Total Profit!
""")
print("✅ Problem Setup Done!")

BUSINESS PROBLEM: Factory Production Optimization

A factory produces two products: A and B

Product A:
- Profit per unit: Rs. 500
- Machine hours required: 2
- Labor hours required: 4

Product B:
- Profit per unit: Rs. 400
- Machine hours required: 5
- Labor hours required: 3

Constraints:
- Total Machine hours available: 40
- Total Labor hours available: 36
- Max Product A: 10 units
- Max Product B: 8 units

GOAL: Maximize Total Profit!

✅ Problem Setup Done!


In [4]:
# STEP 4 - LINEAR PROGRAMMING MODEL

# Create Problem
prob = pulp.LpProblem("Factory_Profit_Maximization", pulp.LpMaximize)

# Decision Variables
A = pulp.LpVariable("Product_A", lowBound=0, cat='Integer')
B = pulp.LpVariable("Product_B", lowBound=0, cat='Integer')

# Objective Function (Maximize Profit)
prob += 500 * A + 400 * B, "Total_Profit"

# Constraints
prob += 2 * A + 5 * B <= 40, "Machine_Hours"
prob += 4 * A + 3 * B <= 36, "Labor_Hours"
prob += A <= 10, "Max_Product_A"
prob += B <= 8, "Max_Product_B"

print("✅ Optimization Model Created!")
print("\nProblem:")
print(prob)

✅ Optimization Model Created!

Problem:
Factory_Profit_Maximization:
MAXIMIZE
500*Product_A + 400*Product_B + 0
SUBJECT TO
Machine_Hours: 2 Product_A + 5 Product_B <= 40

Labor_Hours: 4 Product_A + 3 Product_B <= 36

Max_Product_A: Product_A <= 10

Max_Product_B: Product_B <= 8

VARIABLES
0 <= Product_A Integer
0 <= Product_B Integer



In [5]:
# STEP 5 - SOLVE THE PROBLEM

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print("="*50)
print("OPTIMIZATION RESULTS")
print("="*50)
print(f"Status: {pulp.LpStatus[prob.status]}")
print(f"\nOptimal Solution:")
print(f"Product A units to produce: {int(A.value())}")
print(f"Product B units to produce: {int(B.value())}")
print(f"\nMaximum Profit: Rs. {int(pulp.value(prob.objective))}")
print("="*50)

OPTIMIZATION RESULTS
Status: Optimal

Optimal Solution:
Product A units to produce: 6
Product B units to produce: 4

Maximum Profit: Rs. 4600


In [6]:
# STEP 6 - CONSTRAINTS ANALYSIS

data = {
    'Constraint': ['Machine Hours', 'Labor Hours', 
                   'Max Product A', 'Max Product B'],
    'Available': [40, 36, 10, 8],
    'Used': [
        2*A.value() + 5*B.value(),
        4*A.value() + 3*B.value(),
        A.value(),
        B.value()
    ],
    'Remaining': [
        40 - (2*A.value() + 5*B.value()),
        36 - (4*A.value() + 3*B.value()),
        10 - A.value(),
        8 - B.value()
    ]
}

df = pd.DataFrame(data)
print("Constraints Analysis:")
print(df.to_string(index=False))

Constraints Analysis:
   Constraint  Available  Used  Remaining
Machine Hours         40  32.0        8.0
  Labor Hours         36  36.0        0.0
Max Product A         10   6.0        4.0
Max Product B          8   4.0        4.0


In [ ]:
# STEP 7 - VISUALIZATION

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 - Production Plan
products = ['Product A', 'Product B']
units = [int(A.value()), int(B.value())]
profits = [500*int(A.value()), 400*int(B.value())]

bars = axes[0].bar(products, units, color=['skyblue', 'lightgreen'], 
                    edgecolor='black', width=0.4)
axes[0].set_title('Optimal Production Plan')
axes[0].set_ylabel('Units to Produce')
for bar, unit in zip(bars, units):
    axes[0].text(bar.get_x() + bar.get_width()/2, 
                bar.get_height() + 0.1, 
                str(unit), ha='center', fontweight='bold')

# Plot 2 - Resource Usage
constraints = ['Machine\nHours', 'Labor\nHours']
available = [40, 36]
used = [2*A.value() + 5*B.value(), 4*A.value() + 3*B.value()]

x = np.arange(len(constraints))
width = 0.35
axes[1].bar(x - width/2, available, width, label='Available', 
            color='lightcoral', edgecolor='black')
axes[1].bar(x + width/2, used, width, label='Used', 
            color='steelblue', edgecolor='black')
axes[1].set_title('Resource Usage')
axes[1].set_ylabel('Hours')
axes[1].set_xticks(x)
axes[1].set_xticklabels(constraints)
axes[1].legend()

plt.tight_layout()
plt.savefig('task4_optimization.png')
plt.show()
print("✅ Visualizations Done!")